## Providing Summaries and Timestamps
Exploring how good Claude is at summarizing a full length professional tennis match from youtube, and providing timestamps for certain key moments in the match.

In [1]:
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

# Get the API key
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

# Verify it loaded
if anthropic_api_key:
    print("API key loaded successfully!")
else:
    print("API key not found!")

API key loaded successfully!


In [2]:
from langchain_anthropic import ChatAnthropic

llm_claude_opus = ChatAnthropic(model='claude-opus-4-5')
llm_claude_sonnet = ChatAnthropic(model='claude-sonnet-4-5')
llm_claude_haiku = ChatAnthropic(model='claude-haiku-4-5')

In [3]:
from langchain_openai import ChatOpenAI

llm_gpt_4 = ChatOpenAI(model="gpt-4.1-mini")

In [ ]:
from langchain_community.document_loaders import YoutubeLoader

# 2025 Roland-Garros Final - Sinner vs Alcaraz
# https://youtu.be/ckbX699wngs
loader = YoutubeLoader.from_youtube_url(
    "https://www.youtube.com/watch?v=ckbX699wngs", add_video_info=False
)
docs=loader.load()

transcript=docs[0].page_content

In [ ]:
from langchain_core.prompts import PromptTemplate

prompt_template = """
You are a professional tennis analyst specializing in match analysis and commentary.

Your task: Analyze the provided tennis match transcript and create a structured summary.

TRANSCRIPT:
{video_transcript}

OUTPUT REQUIREMENTS:
1. Match Summary (50-100 words MAX. DO NOT go over 100 words):
   - Players and final score
   - Key turning points
   - Notable performance highlights
   - Match significance/context

2. Significant Moments (with timestamps):
   - Match points
   - Break points at crucial moments
   - Exceptional rallies or shots
   - Momentum shifts
   - Set-defining games

FORMAT:
The match summary should be in paragraph form, to be insert into an HTML web page.
The timestamps should be in MM:SS or HH:MM:SS format and will be listed after the match summary.

TONE: Professional but engaging, suitable for tennis fans.
"""

prompt = PromptTemplate(
    input_variables=["video_transcript"],
    template=prompt_template,
)


In [ ]:
chain = prompt | llm_claude_sonnet
result = chain.invoke({"video_transcript":docs}).content

In [9]:
print(result)

# ROLAND GARROS FINAL MATCH ANALYSIS

## Match Summary

**Carlos Alcaraz defeats Jannik Sinner 6-7(4), 4-6, 6-4, 6-7(6), 7-6(7)**

In one of the most extraordinary Grand Slam finals in recent memory, Carlos Alcaraz successfully defended his Roland Garros title after an epic 5 hours and 29 minutes battle against world number one Jannik Sinner. The match marked the first major final between two players born in the 2000s and lived up to its billing as a generational clash. Sinner dominated the opening two sets with clinical baseline play and exceptional return quality, holding three championship points at 6-3 in the fourth set. However, Alcaraz produced a miraculous escape, winning nine consecutive points to force a deciding set. The Spaniard's forehand, which had been inconsistent early, became a devastating weapon in the final stages. In an unprecedented finish, the match became the first Roland Garros final decided by a fifth-set tiebreak, with Alcaraz proving utterly dominant, winning

In [ ]:
chain = prompt | llm_claude_haiku
result = chain.invoke({"video_transcript":docs}).content

### Testing and exploring Youtube API with Google's official API and Python Client

In [1]:
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

# Get the API key
google_api_key = os.getenv("GOOGLE_API_KEY")

# Verify it loaded
if google_api_key:
    print("API key loaded successfully!")
else:
    print("API key not found!")

API key loaded successfully!


In [ ]:
from googleapiclient.discovery import build

yt = build("youtube", "v3", developerKey=os.getenv("GOOGLE_API_KEY"))

[m for m in dir(yt) if not m.startswith("_")]   # resources: videos, search, channels...
help(yt.videos().list)                           # parameters for one method


In [ ]:
# List JSON contents of a particular video
yt.videos().list(part="snippet,contentDetails,statistics", id="ckbX699wngs").execute()


{'kind': 'youtube#videoListResponse',
 'etag': 'uMzoTJEM5o-DHKoEMV2vcbNI-lc',
 'items': [{'kind': 'youtube#video',
   'etag': 'u4frnX0JHmw3R_roQCT3PYcpim4',
   'id': 'ckbX699wngs',
   'snippet': {'publishedAt': '2025-07-02T07:00:51Z',
    'channelId': 'UCF3K1Jf8hjFW8qliei8fQ3A',
    'title': 'Full match Alcaraz vs Sinner final | Roland-Garros 2025',
    'description': "Watch the 2025 Men's singles final between Carlos Alcaraz and Jannik Sinner in full.\n\n00:00:00 | Final preview\n00:09:40 | First set\n01:15:06 | Second set\n02:24:35 | Third set\n03:14:37 | Fourth set\n04:23:10 | Fifth set\n05:30:29 | Super tiebreak\n05:38:53 | Championship point\n05:43:00 | Trophy ceremony\n\nWelcome to the official Roland-Garros YouTube channel! \n\nFollow us on social media! \nFacebook: https://www.facebook.com/RolandGarros\nX: https://twitter.com/rolandgarros \nInstagram: https://www.instagram.com/rolandgarros\nTikTok: https://www.tiktok.com/@rolandgarros \nWhatsApp: https://bit.ly/WhatsAppRolandGa

In [6]:
# Function to lookup a channel's uploads playlist, and page through it with playlistItems.list
def channel_videos(yt, handle):
    ch = yt.channels().list(part="contentDetails", forHandle=handle).execute()
    uploads = ch["items"][0]["contentDetails"]["relatedPlaylists"]["uploads"]

    videos, token = [], None
    while True:
        res = yt.playlistItems().list(
            part="snippet", playlistId=uploads, maxResults=50, pageToken=token
        ).execute()
        for item in res["items"]:
            s = item["snippet"]
            videos.append({"id": s["resourceId"]["videoId"], "title": s["title"], "published": s["publishedAt"]})
        token = res.get("nextPageToken")
        if not token:
            return videos

In [7]:
vids = channel_videos(yt, "@WTA")
len(vids), vids[:3]

(17649,
 [{'id': '4By6Mjm58-w',
   'title': 'The top 8 doubles teams play finish the thought 💬 #tennis',
   'published': '2026-11-11T17:00:29Z'},
  {'id': 'OXog41-ZcPQ',
   'title': 'Doubles: Finish The Thought | FAGE WTA Finals',
   'published': '2026-11-11T17:00:11Z'},
  {'id': 'EfvTDw2LLnc',
   'title': '“The crowd was INSANE tonight!” 🗣️ #tennis #sports',
   'published': '2026-11-11T06:28:05Z'}])

The YouTube API counts usage in quota units, not tokens. That fetch used about 354 units:

channels.list to find the uploads playlist: 1 unit
playlistItems.list: 17,649 ÷ 50 = 353 pages at 1 unit each, so 353 units
Your earlier videos.list test added 1 more.

The default daily limit is 10,000 units, and it resets at midnight Pacific time. You've used about 3.5%, so you could pull the whole channel again about 28 times a day.

To see your actual usage, go to Google Cloud Console → APIs & Services → YouTube Data API v3 → Quotas & System Limits.

In [9]:
# Save the list of videos from the channel fetch locally via JSON
import json
from pathlib import Path

cache = Path("../data/wta_channel_videos.json")

cache.parent.mkdir(exist_ok=True)
cache.write_text(json.dumps(vids, indent=2, ensure_ascii=False), encoding="utf-8")

# if cache.exists():
#     vids = json.loads(cache.read_text(encoding="utf-8"))
# else:
#     vids = channel_videos(yt, "@TennisChannel")
#     cache.parent.mkdir(exist_ok=True)
#     cache.write_text(json.dumps(vids, indent=2, ensure_ascii=False), encoding="utf-8")

len(vids)


17649

### Filter videos for full matches

In [ ]:
import json
from pathlib import Path

# Load the videos
vids = json.loads(Path("../data/wta_channel_videos.json").read_text(encoding="utf-8"))

In [10]:
full_matches = [v for v in vids if "full match" in v["title"].lower()]
len(full_matches), full_matches[:5]

(212,
 [{'id': '5mUD3y5e9Sw',
   'title': 'The Full 2026 Monterrey Final 🏆 | Diane Parry vs. Elise Mertens | WTA Full Match',
   'published': '2026-10-08T20:00:23Z'},
  {'id': 'JejuoyE8jWg',
   'title': 'The FULL Stearns/Kostyuk vs. Williams/Williams 2026 Cincinnati Round 1 Showdown 💫 | WTA Full Match',
   'published': '2026-10-05T22:46:06Z'},
  {'id': 'DBf0b7Qh6DU',
   'title': 'An All-American Final 🏆 | Coco Gauff vs. Jessica Pegula 2026 Cincinnati Final | WTA Full Match',
   'published': '2026-10-02T20:00:39Z'},
  {'id': 'xbXtUoodm-Q',
   'title': 'The THRILLING Iga Swiatek vs. Elena Rybakina 2026 Toronto Final 🏆 | WTA Full Match',
   'published': '2026-09-22T20:00:06Z'},
  {'id': '8Rz_joxZEWU',
   'title': 'The FULL Jessica Pegula vs. Alexandra Eala 2026 Washington, DC Final 🏆 | WTA Full Match',
   'published': '2026-09-11T20:33:35Z'}])